In [ ]:
import pandas as pd

imdb = pd.read_csv(r"C:\Users\srija\OneDrive\Desktop\machine learning\DEEP LEARNING\datasets\IMDB Dataset.csv")

In [ ]:
imdb.shape

In [ ]:
imdb.head(5)

In [ ]:
imdb.isnull().sum()
imdb.drop_duplicates(inplace=True)

In [ ]:
imdb.shape

## Text Pre-Processing

In [ ]:
# 1. Convert to lowercase
# 2. Remove URLs (http, https)
# 3. Remove HTML
# 4. Remove Punctuations(.,?%#""#)
# 5. Remove stop-words (a, and, the .... etc)
# 6. Stemming and Limitization (Return basic forms)
# 7. Encode Target values (positve, negative, neutral)
# 8. Vectorizations (convert to numbers) -> TF-IDF

In [ ]:
# Converting lowercases
imdb["review"] = imdb["review"].str.lower()

In [ ]:
# Removing URLs
import re

# sample_text = "abc is the word, abc" => convert abc = xyz
# new_text = re.sub("abc", "xyx", sample_text)
# new_text = xyz is the word, xyz

def remove_urls(text):
    text = re.sub(r"http\S+" ," ", text) #(pattern, replace, string)
    return text

imdb["review"] = imdb["review"].apply(remove_urls)

In [ ]:
# Removing Punctuations

def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

imdb["review"] = imdb["review"].apply(remove_punctuations)

In [ ]:
# Removing HTML tags

def remove_htmlTags(text):
    text = re.sub(r"<.*?>", "", text)
    return text

imdb["review"] = imdb["review"].apply(remove_htmlTags)

In [ ]:
# Remove stop-words

import nltk
nltk.download("punkt") # sentence tokenizer
nltk.download("punkt_tab") 
nltk.download("stopwords")

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# sample_text = "I want to be rich"
# tokens = word_tokenize(sample_text)
# print(tokens)

In [ ]:
# Stop Words
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word is stop_words:
            text.replace(word, "")

    return text

imdb["review"] = imdb["review"].apply(remove_stopwords)

In [ ]:
imdb.head()

In [ ]:
# Stemming running -> run, played -> play
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    
    return " ".join(stemmed_words)
imdb["review"] = imdb["review"].apply(stemming)

In [ ]:
# Encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

imdb["sentiment"] = le.fit_transform(imdb["sentiment"])

In [ ]:
y = imdb["sentiment"]

In [ ]:
# Vectorizations -> convert the tokens into numericals for processing
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=9000) # max_featres=9000, I DIDNT DO IT MUST DO IT.

X = tf.fit_transform(imdb["review"])

In [ ]:
print(X)

### Dataset and Data Loaders

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print(f"X_train -> {X_train.shape}")
print(f"X_test -> {X_test.shape}")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
import numpy as np
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

### Build RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN LAYER
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True) #prebuilt RNN

        # FUlly ConneCted LayerR
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape(num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out,_ = self.rnn(x, h0) # 1st value -> hidden state of all the timesteps. (batch, seq_len, hidden_size)
                                # 2nd value -> final hidden state of last timesteps.
        
        out = self.fc(out[:, -1])
        return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the Model

In [ ]:
# unsqueeze -> add 1 dimension, and squeeze reduce 1 dimension
epochs = 10
for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # Add singleton directions

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batchsize, )

        loss = criterion(outputs, yb) # Computer Loss
        loss.backward() # back Prop
        optimizer.step() # weights update
    
    print(f"epoch = {epoch+1}/{epochs} & loss = {loss.item()}")

In [ ]:
# Evaluations
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

print(f"accuracy = {correct_vals/tot_vals*100}")